### 🔗 Links and Resources
| Method / Config | Description |
|-----------------|-------------|
| [DataFrame.repartition()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html) | Returns a new DataFrame with exactly N partitions (full shuffle) |
| [DataFrame.coalesce()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html) | Reduces partition count without a full shuffle |
| [DataFrame.write.partitionBy()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html) | Writes data to disk split into subdirectories by column value |
| [DataFrame.write.bucketBy()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.bucketBy.html) | Writes data into a fixed number of hash buckets (requires `saveAsTable`) |
| [DataFrame.write.sortBy()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.sortBy.html) | Sorts data within each bucket before writing |
| [DataFrame.explain()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) | Prints the physical plan — look for `Exchange` nodes indicating shuffles |
| `spark.sql.shuffle.partitions` | Controls the number of partitions after a shuffle (default 200) |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

### 📌 Creating Sample Data

We create a sales dataset with columns suitable for demonstrating both
**partitioning** (by `region` and `year`) and **bucketing** (by `customer_id`).

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F

sales_schema = StructType([
    StructField("sale_id",     IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("region",      StringType(),  True),
    StructField("year",        IntegerType(), True),
    StructField("product",     StringType(),  True),
    StructField("amount",      DoubleType(),  True),
])

sales_data = [
    (1,  101, "North", 2023, "Widget A",  250.0),
    (2,  102, "South", 2023, "Widget B",  180.0),
    (3,  103, "East",  2023, "Gadget C",  420.0),
    (4,  101, "North", 2024, "Widget A",  310.0),
    (5,  104, "West",  2023, "Part E",     90.0),
    (6,  102, "South", 2024, "Gadget D",  530.0),
    (7,  105, "East",  2024, "Widget B",  200.0),
    (8,  103, "North", 2024, "Part E",    115.0),
    (9,  104, "West",  2023, "Gadget C",  640.0),
    (10, 105, "South", 2023, "Widget A",  175.0),
    (11, 101, "East",  2024, "Gadget D",  390.0),
    (12, 102, "North", 2023, "Part E",     60.0),
]

df_sales = spark.createDataFrame(sales_data, schema=sales_schema)

print(f"Total rows: {df_sales.count()}  |  Partitions: {df_sales.rdd.getNumPartitions()}")
display(df_sales)

### 📌 In-Memory Partitioning with repartition() and coalesce()

**In-memory partitions** control how Spark splits a DataFrame across executor tasks.
They exist only during computation — nothing is written to disk.

| Method | Shuffle? | Use case |
|--------|----------|----------|
| `repartition(n)` | Yes | Increase or evenly redistribute partitions |
| `repartition(n, col)` | Yes | Co-locate rows with the same column value |
| `coalesce(n)` | No (usually) | Reduce partitions cheaply |

In [ ]:
# Default partition count
print(f"Default partitions : {df_sales.rdd.getNumPartitions()}")

# Repartition to 4 — full shuffle, even distribution
df_4 = df_sales.repartition(4)
print(f"After repartition(4): {df_4.rdd.getNumPartitions()}")

# Repartition by column — all rows for the same region land on the same partition
df_by_region = df_sales.repartition(4, "region")
print(f"After repartition(4, 'region'): {df_by_region.rdd.getNumPartitions()}")

# Coalesce — merge partitions without a full shuffle (only reduces, never increases)
df_coalesced = df_4.coalesce(2)
print(f"After coalesce(2)   : {df_coalesced.rdd.getNumPartitions()}")

### 📌 Inspecting Partition Contents

`glom()` converts each partition into a list so we can see which rows
ended up together. After `repartition('region')` every row in a partition
should share the same `region` value.

In [ ]:
# Show region distribution across the 4 partitions
partition_contents = df_by_region.select("sale_id", "region").rdd.glom().collect()

for i, partition in enumerate(partition_contents):
    regions = [row["region"] for row in partition]
    print(f"Partition {i} ({len(partition)} rows): {regions}")

### 📌 On-Disk Partitioning with write.partitionBy()

`partitionBy()` writes data into a **directory hierarchy** based on column values:
```
output/
  region=North/
    year=2023/part-00000.snappy.parquet
    year=2024/part-00000.snappy.parquet
  region=South/
    ...
```
When you later read and filter on `region` or `year`, Spark **skips entire subdirectories**
(partition pruning) without scanning unrelated files.

In [ ]:
output_path = "/tmp/sales_partitioned"

# Write partitioned by region then year
(
    df_sales
    .write
    .mode("overwrite")
    .partitionBy("region", "year")
    .parquet(output_path)
)

print(f"Written to: {output_path}")
print("Directory layout (region= subdirs):")

# List the top-level partition directories
import subprocess
result = spark.sql(f"LIST '{output_path}'")
display(result)

### 📌 Partition Pruning in Action

Reading with a filter on the partition column lets Spark skip all other directories.
The physical plan will show a `PartitionFilters` predicate pushed down to the scan.

In [ ]:
# Read back only the North region — Spark skips South/East/West directories entirely
df_north = spark.read.parquet(output_path).filter(F.col("region") == "North")

print("--- Rows returned ---")
display(df_north)

print("\n--- Physical plan (look for PartitionFilters) ---")
df_north.explain(extended=False)

### 📌 What is Bucketing?

**Bucketing** organises data into a **fixed number of files** using a hash of a column value.
All rows where `hash(customer_id) % num_buckets == k` land in bucket `k`.

Key differences from `partitionBy`:

| Feature | partitionBy | bucketBy |
|---------|-------------|----------|
| Storage layout | One subdir per distinct value | Fixed number of files |
| Good for | Low-cardinality columns (region, year) | High-cardinality columns (customer_id, order_id) |
| Join optimisation | Partition pruning | Eliminates shuffle on bucketed key |
| Requires `saveAsTable`? | No | Yes |

> **Note:** `bucketBy` requires writing as a Hive-compatible managed or external table.
> On Databricks, use `spark.sql.warehouse.dir` or Unity Catalog managed tables.

In [ ]:
# Create a database/schema to hold the bucketed tables
spark.sql("CREATE DATABASE IF NOT EXISTS demo_bucketing")
spark.sql("USE demo_bucketing")

# Write sales table bucketed by customer_id into 4 buckets, sorted within each bucket
(
    df_sales
    .write
    .mode("overwrite")
    .bucketBy(4, "customer_id")
    .sortBy("customer_id")
    .saveAsTable("demo_bucketing.sales_bucketed")
)

print("Bucketed table written. Verifying row count:")
spark.sql("SELECT COUNT(*) AS total FROM demo_bucketing.sales_bucketed").show()

### 📌 Bucket Join — Eliminating the Shuffle

When **both** sides of a join are bucketed on the same column with the **same number
of buckets**, Spark can pair up matching buckets directly — no shuffle required.
The physical plan will show `SortMergeJoin` without any `Exchange` node.

In [ ]:
from pyspark.sql.types import DoubleType

# Create a second bucketed table: customer credit limits
customers_data = [
    (101, "Alice",   5000.0),
    (102, "Bob",     3500.0),
    (103, "Carol",   8000.0),
    (104, "Dave",    2000.0),
    (105, "Eve",     6500.0),
]

customers_schema = StructType([
    StructField("customer_id",  IntegerType(), True),
    StructField("name",         StringType(),  True),
    StructField("credit_limit", DoubleType(),  True),
])

df_customers = spark.createDataFrame(customers_data, schema=customers_schema)

(
    df_customers
    .write
    .mode("overwrite")
    .bucketBy(4, "customer_id")   # Same column, same bucket count
    .sortBy("customer_id")
    .saveAsTable("demo_bucketing.customers_bucketed")
)

print("Customers bucketed table written.")

In [ ]:
# Read both bucketed tables and join on customer_id
df_sales_t    = spark.table("demo_bucketing.sales_bucketed")
df_customers_t = spark.table("demo_bucketing.customers_bucketed")

df_joined = df_sales_t.join(df_customers_t, on="customer_id", how="inner")

print("--- Join result ---")
display(df_joined.select("customer_id", "name", "region", "year", "amount", "credit_limit"))

print("\n--- Physical plan (no Exchange = no shuffle!) ---")
df_joined.explain(extended=False)

### 📌 Combining partitionBy and bucketBy

You can layer both strategies: **partition by a low-cardinality column** for pruning,
and **bucket by a high-cardinality column** within each partition for shuffle-free joins.

In [ ]:
(
    df_sales
    .write
    .mode("overwrite")
    .partitionBy("year")           # Subdirectory per year (low cardinality)
    .bucketBy(4, "customer_id")    # 4 bucket files inside each year dir
    .sortBy("customer_id")
    .saveAsTable("demo_bucketing.sales_partitioned_bucketed")
)

print("Combined table written.")
spark.sql("""
    SELECT year, COUNT(*) AS rows
    FROM demo_bucketing.sales_partitioned_bucketed
    GROUP BY year
    ORDER BY year
""").show()

### 📌 Inspecting Table Metadata

`DESCRIBE EXTENDED` reveals the bucketing configuration stored in the metastore.

In [ ]:
desc = spark.sql("DESCRIBE EXTENDED demo_bucketing.sales_bucketed")

# Filter to the bucketing-related rows
display(
    desc.filter(
        F.col("col_name").isin(
            "Num Buckets", "Bucket Columns", "Sort Columns",
            "Type", "Provider", "Statistics"
        )
    )
)

### 📌 Controlling Shuffle Partitions

`spark.sql.shuffle.partitions` sets the default number of partitions produced
after any shuffle operation (joins, aggregations). The default of **200** is often
too high for small datasets and too low for very large ones.

In [ ]:
print("Default shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Reduce to 4 for this small dataset
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_agg = df_sales.groupBy("region").agg(F.sum("amount").alias("total_sales"))
print(f"\nPartitions after groupBy with shuffle.partitions=4: {df_agg.rdd.getNumPartitions()}")
display(df_agg)

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "200")
print("\nspark.sql.shuffle.partitions restored to 200")

### 📌 Cleanup

Drop the demo tables and database created during this session.

In [ ]:
spark.sql("DROP TABLE IF EXISTS demo_bucketing.sales_bucketed")
spark.sql("DROP TABLE IF EXISTS demo_bucketing.customers_bucketed")
spark.sql("DROP TABLE IF EXISTS demo_bucketing.sales_partitioned_bucketed")
spark.sql("DROP DATABASE IF EXISTS demo_bucketing CASCADE")

print("Cleanup complete.")